In [ ]:
# -*- coding: utf-8 -*-
"""Matrix_Actual_Values.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1uWRzKABpo0tBbppDvsixCQIlz73cgLfC
"""

import pandas as pd
import numpy as np

# Function 3: Calculate Actual Positions and Velocities (with ego vehicle reference)
def calculate_actual_positions_and_velocities(tracks_df, ego_id=5):
    def calculate_actual_data(row):
        # Define the surrounding vehicles order
        surrounding_ids = [
            row['leftPrecedingId'], row['precedingId'], row['rightPrecedingId'],
            row['leftAlongsideId'], row['id'], row['rightAlongsideId'],
            row['leftFollowingId'], row['followingId'], row['rightFollowingId']
        ]

        surrounding_data = {}
        for i, s_id in enumerate(surrounding_ids):
            if s_id > 0:  # Vehicle exists
                # Match both ID and frame to get accurate data
                s_vehicle = tracks_df[(tracks_df['id'] == s_id) & (tracks_df['frame'] == row['frame'])]
                if not s_vehicle.empty:
                    surrounding_data[f'vehicle_{i+1}_actual_x'] = s_vehicle['x'].iloc[0]
                    surrounding_data[f'vehicle_{i+1}_actual_y'] = s_vehicle['y'].iloc[0]
                    surrounding_data[f'vehicle_{i+1}_actual_vx'] = s_vehicle['xVelocity'].iloc[0]
                    surrounding_data[f'vehicle_{i+1}_actual_vy'] = s_vehicle['yVelocity'].iloc[0]
                else:
                    # Assign default values if vehicle not found at the same frame
                    surrounding_data[f'vehicle_{i+1}_actual_x'] = row['x'] + np.random.uniform(0, 0.01)
                    surrounding_data[f'vehicle_{i+1}_actual_y'] = row['y'] + np.random.uniform(0, 0.01)
                    surrounding_data[f'vehicle_{i+1}_actual_vx'] = row['xVelocity'] + np.random.uniform(0, 0.01)
                    surrounding_data[f'vehicle_{i+1}_actual_vy'] = row['yVelocity'] + np.random.uniform(0, 0.01)
            else:  # No vehicle or invalid ID
                surrounding_data[f'vehicle_{i+1}_actual_x'] = row['x'] + np.random.uniform(0, 0.01)
                surrounding_data[f'vehicle_{i+1}_actual_y'] = row['y'] + np.random.uniform(0, 0.01)
                surrounding_data[f'vehicle_{i+1}_actual_vx'] = row['xVelocity'] + np.random.uniform(0, 0.01)
                surrounding_data[f'vehicle_{i+1}_actual_vy'] = row['yVelocity'] + np.random.uniform(0, 0.01)

        return surrounding_data

    # Apply the function for all rows
    actual_data = tracks_df.apply(calculate_actual_data, axis=1)
    actual_data_df = pd.DataFrame(actual_data.tolist())
    tracks_df = pd.concat([tracks_df, actual_data_df], axis=1)
    return tracks_df


# Function 4: Reshape into a single column matrix
def reshape_to_vehicle_matrix(tracks_df):
    matrix_data = []
    for _, row in tracks_df.iterrows():
        vehicle_data = []
        for vehicle_num in range(1, 10):  # Vehicles 1-9
            vehicle_features = [
                row.get(f"vehicle_{vehicle_num}_actual_x", -1),
                row.get(f"vehicle_{vehicle_num}_actual_y", -1),
                row.get(f"vehicle_{vehicle_num}_actual_vx", 0),
                row.get(f"vehicle_{vehicle_num}_actual_vy", 0),
            ]
            vehicle_data.append(vehicle_features)

        # Flatten into a single row
        matrix_data.append(np.array(vehicle_data).flatten())

    matrix_columns = [f"feature_{i+1}" for i in range(len(matrix_data[0]))]
    vehicle_matrix = pd.DataFrame(matrix_data, columns=matrix_columns)
    return vehicle_matrix

# Function 5: Fill Missing Values
def fill_missing_values(df):
    position_columns = [col for col in df.columns if 'actual_x' in col or 'actual_y' in col]
    velocity_columns = [col for col in df.columns if 'actual_vx' in col or 'actual_vy' in col]

    ego_vx = df['xVelocity']
    ego_vy = df['yVelocity']

    for col in position_columns:
        df[col] = df[col].fillna(-1)

    for col in velocity_columns:
        if 'actual_vx' in col:
            df[col] = df[col].fillna(ego_vx)
        elif 'actual_vy' in col:
            df[col] = df[col].fillna(ego_vy)

    return df

# Function 6: Assign Time Intervals
def assign_time_intervals(df, interval=75):
    df['time'] = 'T1'
    max_frame = df['frame'].max()
    time_labels = [f'T{i+1}' for i in range((max_frame // interval) + 1)]
    for i, label in enumerate(time_labels):
        start_frame = i * interval
        end_frame = start_frame + interval - 1
        df.loc[(df['frame'] >= start_frame) & (df['frame'] <= end_frame), 'time'] = label
    return df

# Main Execution
if __name__ == "__main__":
    # Load the meta and track data
    tracks_df = pd.read_csv('/content/02_tracks.csv')
    meta_df = pd.read_csv("/content/02_tracksMeta.csv")

    # Merge meta features with tracks
    combined_df = tracks_df.merge(meta_df, on='id', how='left')

    # Calculate actual positions and velocities
    combined_df = calculate_actual_positions_and_velocities(combined_df)

    # Fill missing values
    combined_df = fill_missing_values(combined_df)

    # Assign time intervals
    combined_df = assign_time_intervals(combined_df)

    # Reshape to vehicle matrix
    vehicle_matrix = reshape_to_vehicle_matrix(combined_df)

    # Add the matrix as a single column in the original DataFrame
    combined_df['vehicle_matrix'] = vehicle_matrix.apply(lambda row: row.tolist(), axis=1)

    # Save the final processed data
    combined_df.to_csv('processed_traffic_data_with_actual_matrix_and_time2.csv', index=False)

    print("Data preprocessing complete. File saved as 'processed_traffic_data_with_actual_matrix_and_time.csv'.")



Data preprocessing complete. File saved as 'processed_traffic_data_with_actual_matrix_and_time.csv'.
